In [ ]:
# GPU/XLA path setup (run this first, before importing tensorflow/keras/orion)
import os
import sys

_site = next((p for p in sys.path if "site-packages" in p), None)
if _site:
    _nvcc = os.path.join(_site, "nvidia", "cuda_nvcc")
    _nvrtc = os.path.join(_site, "nvidia", "cuda_nvrtc", "lib")

    if os.path.isdir(_nvcc):
        os.environ["XLA_FLAGS"] = "--xla_gpu_cuda_data_dir=" + _nvcc
        _ptxas = os.path.join(_nvcc, "bin")
        os.environ["PATH"] = _ptxas + os.pathsep + os.environ.get("PATH", "")

    if os.path.isdir(_nvrtc):
        old = os.environ.get("LD_LIBRARY_PATH", "")
        os.environ["LD_LIBRARY_PATH"] = _nvrtc + (os.pathsep + old if old else "")

print("XLA_FLAGS:", os.environ.get("XLA_FLAGS", "<unset>"))

In [ ]:
import pandas as pd

from orion.data import load_signal

# 1. Data

In [ ]:
signal_name = 'S-1'

data = load_signal(signal_name)

data.head()

# 2. Pipeline

In [ ]:
from mlblocks import MLPipeline

pipeline_name = 'aer'

pipeline = MLPipeline(pipeline_name)

In [ ]:
hyperparameters = {
    'orion.primitives.aer.AER#1': {
        'epochs': 5,
        'verbose': True
    }
}

pipeline.set_hyperparameters(hyperparameters)

## step by step execution

MLPipelines are compose of a squence of primitives, these primitives apply tranformation and calculation operations to the data and updates the variables within the pipeline. To view the primitives used by the pipeline, we access its `primtivies` attribute. 

The `lstm_dynamic_threshold` contains 7 primitives. we will observe how the `context` (which are the variables held within the pipeline) are updated after the execution of each primitive.

In [ ]:
pipeline.primitives

### time segments aggregate
this primitive creates an equi-spaced time series by aggregating values over fixed specified interval.

* **input**: `X` which is an n-dimensional sequence of values.
* **output**:
    - `X` sequence of aggregated values, one column for each aggregation method.
    - `index` sequence of index values (first index of each aggregated segment).

In [ ]:
context = pipeline.fit(data, output_=0)
context.keys()

In [ ]:
for i, x in list(zip(context['index'], context['X']))[:5]:
    print("entry at {} has value {}".format(i, x))

### SimpleImputer
this primitive is an imputation transformer for filling missing values.
* **input**: `X` which is an n-dimensional sequence of values.
* **output**: `X` which is a transformed version of X.

In [ ]:
step = 1

context = pipeline.fit(**context, output_=step, start_=step)
context.keys()

### MinMaxScaler
this primitive transforms features by scaling each feature to a given range.
* **input**: `X` the data used to compute the per-feature minimum and maximum used for later scaling along the features axis.
* **output**: `X` which is a transformed version of X.

In [ ]:
step = 2

context = pipeline.fit(**context, output_=step, start_=step)
context.keys()

In [ ]:
# after scaling the data between [-1, 1]
# in this example, no change is observed
# since the data was pre-handedly scaled

for i, x in list(zip(context['index'], context['X']))[:5]:
    print("entry at {} has value {}".format(i, x))

### rolling window sequence
this primitive generates many sub-sequences of the original sequence. it uses a rolling window approach to create the sub-sequences out of time series data.

* **input**: 
    - `X` n-dimensional sequence to iterate over.
    - `index` array containing the index values of X.
* **output**:
    - `X` input sequences.
    - `y` target sequences.
    - `index` first index value of each input sequence.
    - `target_index` first index value of each target sequence.

In [ ]:
step = 3

context = pipeline.fit(**context, output_=step, start_=step)
context.keys()

In [ ]:
# after slicing X into multiple sub-sequences
# we obtain a 3 dimensional matrix X where
# the shape indicates (# slices, window size, 1)
# and similarly y is (# slices, target size)

print("X shape = {}\ny shape = {}\nX index shape = {}\ny index shape = {}".format(
    context['X'].shape, context['y'].shape, context['index'].shape, context['y_index'].shape))

y_ = context['y'].copy() # make a copy to copare later

### slice array by dims

this primitive selects a particular channel from `X` to try and reconstruct / predict it.

* **input**:
    * `X` n-dimensional array containing the input sequences.
    * `target_index` which index from `X` to slice out.
* **output**:
    * `y` 1-dimenstional array containing the target sequences.

In [ ]:
step = 4

context = pipeline.fit(**context, output_=step, start_=step)
context.keys()

### AER
this is a hybrid prediction and reconstruction model using LSTM layers. you can read more about it in the [related paper](https://arxiv.org/pdf/2212.13558).

* **input**: 
    - `X` n-dimensional array containing the input sequences for the model.
    - `y` n-dimensional array containing the target sequences for the model.
* **output**: `y_hat` predicted values.

In [ ]:
step = 5

context = pipeline.fit(**context, output_=step, start_=step)
context.keys()

In [ ]:
ry, fy = context['y'][:, 0], context['y'][:, -1]

for i, y, y_hat in list(zip(context['y_index'], fy, context['fy_hat']))[:5]:
    print("entry at {} has value {}, predicted value {}".format(i, y, y_hat))

### score anomalies

this primitive computes an array of errors comparing predictions and expected output.

* **input**: 
    - `y` ground truth.
    - `y_hat` reconstructed values.
    - `fy_hat` forward prediction values.
    - `ry_hat` reverse prediction values.
* **output**: `errors` array of errors.

In [ ]:
step = 6

context = pipeline.fit(**context, output_=step, start_=step)
context.keys()

In [ ]:
for i, e in list(zip(context['y_index'], context['errors']))[:5]:
    print("entry at {} has error value {:.3f}".format(i, e))

In [ ]:
import matplotlib.pyplot as plt

fig, axs = plt.subplots(2, 1, figsize=(10, 3))

ax1, ax2 = axs
ax1.plot(fy, label='original')
ax1.plot(context['fy_hat'], label='predicted')
ax1.set_title('fy')

ax2.plot(ry, label='original')
ax2.plot(context['ry_hat'], label='predicted')
ax2.set_title('ry')

handles, labels = ax1.get_legend_handles_labels()
fig.legend(handles, labels, ncols=2, loc='upper right')
plt.tight_layout()
plt.show();

In [ ]:
import matplotlib.pyplot as plt

fig = plt.figure(figsize=(10, 2))
ax = fig.gca()

t = data['timestamp'].values

data.plot(x='timestamp', y='value', ax=ax)
plt.plot(t[100:], context['fy_hat'], label='fy')
plt.plot(t[:-100], context['ry_hat'], label='ry')

plt.plot(t[:-1], context['errors'], label='errors')
plt.legend(ncols=4)

plt.show();

### find anomalies
this primitive extracts anomalies from sequences of errors following the approach explained in the [related paper](https://arxiv.org/pdf/1802.04431.pdf).

* **input**: 
    - `errors` array of errors.
    - `target_index` array of indices of errors.
* **output**: `y` array containing start-index, end-index, score for each anomalous sequence that was found.

In [ ]:
step = 7

context = pipeline.fit(**context, output_=step, start_=step)
context.keys()

In [ ]:
pd.DataFrame(context['y'], columns=['start', 'end', 'severity'])